In [ ]:
%pip install ipykernel
%pip install tushare
%pip install pandas
%pip install matplotlib
%pip install pyecharts

In [2]:
import os
import tushare as ts
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import pyecharts as pc


In [ ]:
#解决matplotlib画图不能显示中文字体的问题（临时解决）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  

获取token

In [ ]:
# 从环境变量中读取 token
token = os.environ.get('TUSHARE_TOKEN')
# 初始化 pro 接口 [2]
pro = ts.pro_api(token)

连接数据库

In [ ]:
#创建数据库文件连接
conn1=sqlite3.connect("D:\\Cursor项目统一管理\\Trading_Strategy_Backtesting\\data\\ETF.db")
conn1.close#close没有加括号的话close(),这个函数不被执行

把数据表存进SQL

In [ ]:
#存数据
#获取ETF基本信息
df = pro.fund_basic()
#获取日线数据
data=pro.fund_daily(ts_code='513310.SH', start_date='19990101', end_date='20260325')
data.to_sql('中韩半导体ETF日线数据',conn1,if_exists='replace',index=False)
conn1.close()


从sql拿出数据表

In [ ]:
#取数据
data_read=pd.read_sql("SELECT*FROM 中韩半导体ETF日线数据 ORDER BY trade_date ASC",conn1,parse_dates=['trade_date'])
#index_col='trade_date'
conn1.close()


计算指标

In [ ]:
data_read['MA10'] = data_read['close'].rolling(window=10).mean()

编写策略函数
!注意对齐K线图的坐标轴数据数量和数据格式，否则会出现冲突,另外别用日期作为对齐轴

In [ ]:
buy_price=[None]
sell_price=[None]

for i in range(1,len(data_read)):
    cur_data=data_read['close'][i];
    cur_madata=data_read['MA10'][i];
    his_data=data_read['close'][i-1];
    his_madata=data_read['MA10'][i-1];
    if cur_data > cur_madata and his_data < his_madata:
        buy_price.append(data_read['close'][i])
    else:
        buy_price.append(None)
    if cur_data < cur_madata and his_data > his_madata:
        sell_price.append(data_read['close'][i])
    else:
        sell_price.append(None)
    
    
print(buy_price)
print(len(buy_price),len(data_read),len(sell_price))

画K线图

In [ ]:
# 1. 准备数据：[开盘, 收盘, 最低, 最高]
# 假设从之前的 data_read 提取
x_data = data_read['trade_date'].astype(str).tolist()
y_data = data_read[['open', 'close', 'low', 'high']].values.tolist()
x_madata=data_read['MA10']

# 2. 构建 K 线图
kline = (
    pc.charts
    .Kline(init_opts=pc.options.InitOpts( theme=pc.globals.ThemeType.DARK,width="100%", height="100vh",bg_color='#1d1b1c'))
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="K线图",
        y_axis=y_data,
        # 设置红绿颜色样式
        itemstyle_opts = pc.options.ItemStyleOpts(
            color="red",    # 阳线（涨）
            color0="green",   # 阴线（跌）
            border_color="#8A0000",
            border_color0="#008F28",
        ))
    .set_global_opts(
        title_opts=pc.options.TitleOpts(title="中韩半导体ETF"),
        # 启用你之前需要的缩放功能
        datazoom_opts=[pc.options.DataZoomOpts(type_="slider", min_span=3,range_start=90,range_end=100 ), 
                       pc.options.DataZoomOpts(type_="inside",is_zoom_on_mouse_wheel=True,       # 鼠标滚轮缩放（跟随鼠标位置）
                                               is_move_on_mouse_move=True,        # 鼠标拖拽移动（跟随鼠标）
                                               is_move_on_mouse_wheel=True,       # 鼠标滚轮移动
                                               is_zoom_lock=False,                # 不锁定缩放区域
                                               min_span=5,
                                               )],
        xaxis_opts=pc.options.AxisOpts(splitline_opts=pc.options.SplitLineOpts(is_show=False)), # 去掉 X 轴网格线
        yaxis_opts=pc.options.AxisOpts(is_scale=True,axislabel_opts=pc.options.LabelOpts(
            # 这里的 3 代表强制保留 3 位小数，无论缩放到什么程度都不会改变
                                       formatter=pc.commons.utils.JsCode("function (value) { return value.toFixed(3); }")), # 坐标轴不从0开始
))
)




指标绘制

In [ ]:
# 3. 叠加指标
ma_line=(pc.charts.Line()
.add_xaxis(xaxis_data=x_data)
.add_yaxis(series_name="MA10",
    y_axis=x_madata.tolist(),
    is_smooth=True,        # 线条平滑显示
    is_symbol_show=False,
    linestyle_opts=pc.options.LineStyleOpts(width=2),
    itemstyle_opts = pc.options.ItemStyleOpts(color='yellow')
))
kline.overlap(ma_line)

# 3. 渲染
kline.render("etf_kline.html")

买卖点绘图

In [ ]:


bs_scatter=(pc.charts.Scatter()
.add_xaxis(xaxis_data=x_data)
.add_yaxis(series_name='B点',
y_axis=buy_price,
symbol_size=15,
itemstyle_opts=pc.options.ItemStyleOpts(color='#e06d34')#更改局部颜色
)
.add_yaxis(series_name='S点',
y_axis=sell_price,
symbol_size=15,
itemstyle_opts=pc.options.ItemStyleOpts(color='#1edda2')#更改局部颜色

)
)

kline.overlap(bs_scatter)
kline.render("etf_kline.html")


回测模块

In [ ]:
initial_cash = 1000000  # 设定初始资金 [2]
cash = initial_cash
holdings = 0
net_value_list = []

# 模拟交易循环 [1]
for i in range(len(data_read)):
    current_price = data_read['close'][i]
    
    # 买入逻辑：若有买入信号且当前为空仓 [3], [1]
    if buy_price[i] is not None and holdings == 0:
        holdings = cash / current_price
        cash = 0
    
    # 卖出逻辑：若有卖出信号且当前有持仓 [4], [1]
    elif sell_price[i] is not None and holdings > 0:
        cash = holdings * current_price
        holdings = 0
    
    # 计算当日净值：现金 + 持仓市值 [2]
    daily_net_value = cash + (holdings * current_price)
    net_value_list.append(daily_net_value)

# 将净值曲线存入 DataFrame
data_read['net_value'] = net_value_list

# 计算核心评价指标 [2], [5]
final_value = data_read['net_value'].iloc[-1]
# 累计收益率 = (测试期末资产 - 测试期初资产) / 测试期初资产 [2]
total_return = (final_value - initial_cash) / initial_cash

# 计算最大回撤：从高点回撤最大值 / 高点资产量 [5]
rolling_max = data_read['net_value'].cummax()
drawdown = (data_read['net_value'] - rolling_max) / rolling_max
max_drawdown = drawdown.min()

print(f"回测完成！累计收益率: {total_return:.2%}, 最大回撤: {max_drawdown:.2%}")
